In [ ]:
# Install AutoGluon
!pip install -q ray==2.10.0
!pip install autogluon.tabular
!pip install -U ipywidgets

1. Imports:
* **pandas, lightgbm, xgboost, matplotlib.pyplot, seaborn**: These libraries are used for data manipulation, visualization, and machine learning.
* **%matplotlib inline**: Ensures that plots are displayed inline in notebooks.
* **warnings**: Suppresses certain warnings.
* **re**: Regular expression library, useful for text pattern matching.
* **TabularPredictor from AutoGluon**: A tool for automated machine learning on tabular data.

In [ ]:
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set_style("whitegrid")
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import re
from autogluon.tabular import TabularPredictor

**main_data** and **train_data**: Two datasets will load using pandas.read_csv() from different CSV files. The id column in **train_data** and **test_data ** will drope since it's not needed for training the model.

In [ ]:
# Load the data

main_data = pd.read_csv('/kaggle/input/used-car-price-prediction-dataset/used_cars.csv')
train_data = pd.read_csv('/kaggle/input/playground-series-s4e9/train.csv').drop('id', axis=1)
test_data = pd.read_csv('/kaggle/input/playground-series-s4e9/test.csv').drop('id', axis=1)

* Data Cleaning:
**main_data['price']**: The price column in main_data is cleaned by removing "$" and , characters, then converting the values to float for numerical analysis.
**main_data['milage']**: The milage column is cleaned similarly, removing non-numeric characters like 'mi.' and commas, and then converting it to float.

In [ ]:
main_data['price'] = main_data['price'].str.replace('$', '').str.replace(',', '').astype(float)
main_data['milage'] = main_data['milage'].str.replace('mi.', '').str.replace(',', '').astype(float)

Data Concatenation:
combined_data: The code concatenates train_data and main_data into one combined_data DataFrame. This step is likely for merging the datasets before further analysis or model training.

**Feature Engineering:**
The feature_engineering function is defined to preprocess the dataset:
Handling Missing Values: The 'accident' column is simplified by mapping different accident statuses to 'Yes' or 'No' and filling missing values with 'Yes'.
Grouping Rare Categories: For the 'brand' and 'model' columns, infrequent categories are grouped into 'Other' based on the frequency threshold.
Engine Feature Extraction: The function uses regular expressions (re.search()) to extract information such as horsepower, displacement (in liters), and the number of cylinders from the 'engine' column.
Fuel Type Simplification: The fuel type is simplified into categories like 'Gasoline', 'Diesel', 'Turbocharged', and 'Unknown'.
Transmission Simplification: Transmission types are reduced to either 'Manual', 'Automatic', or 'Unknown'.

In [ ]:
# Concatenating the two DataFrames
combined_data = pd.concat([train_data, main_data], axis=0)

# Displaying the combined DataFrame
combined_data.dtypes

**Model Training:**
AutoGluon TabularPredictor:
The target variable for the prediction is defined as 'price'.
The model is trained on train_data using AutoGluon’s TabularPredictor. AutoGluon automatically selects the best machine learning models based on the data. In this case, two specific models (XGB for XGBoost and GBM for LightGBM) are specified.
The presets='best_quality' option configures the model for high prediction quality. The time_limit=3600 sets a time constraint of 1 hour for model training.

In [ ]:
from autogluon.tabular import TabularPredictor


# Feature engineering function from the notebook
def feature_engineering(data):
    # Handle missing values and simplify the 'accident' column
    data["accident"] = data["accident"].replace({
        "At least 1 accident or damage reported": "Yes",
        "None reported": "No"
    }).fillna("Yes")

    # Group rare brands and models under 'Other'
    for col, replace_threshold in [("brand", 2222), ("model", 200)]:
        counts = data[col].value_counts()
        to_replace = counts[counts < replace_threshold].index
        data[col] = data[col].apply(lambda x: "Other" if x in to_replace else x)

    # Extract engine features (e.g., horsepower, displacement, cylinders)
    def extract_features(engine, pattern, group=1, dtype=float):
        match = re.search(pattern, engine)
        try:
            return dtype(match.group(group)) if match else None
        except (ValueError, TypeError):
            return None

    data['horsepower'] = data['engine'].apply(lambda x: extract_features(x, r'(\d+(\.\d+)?)HP'))
    data['displacement'] = data['engine'].apply(lambda x: extract_features(x, r'(\d+(\.\d+)?)L'))
    data['cylinders'] = data['engine'].apply(lambda x: extract_features(x, r'(\d+)\s*Cyl|I(\d+)|V(\d+)|Straight\s*(\d+)', group=1, dtype=int))
    data['cylinders'] = data['cylinders'].apply(lambda x: int(x) if pd.notna(x) else None)

    # Extract and simplify fuel types
    data['fuel_type'] = data['engine'].apply(lambda x:
                                             'Gasoline' if 'Gasoline' in str(x) else
                                             ('Diesel' if 'Diesel' in str(x) else
                                             ('Turbocharged' if 'Turbo' in str(x) else 'Unknown')))

    # Simplify transmission types
    def simplify_transmission(trans):
        if not isinstance(trans, str):
            return 'Unknown'
        trans = trans.lower()
        if any(keyword in trans for keyword in ['manual', 'auto']):
            return 'Manual' if 'manual' in trans else 'Automatic'
        return 'Unknown'

    data['transmission'] = data['transmission'].apply(simplify_transmission)

    return data



# Apply feature engineering
train_data = feature_engineering(train_data)
test_data = feature_engineering(test_data)

# Specify the label (target) column
label_column = 'price'  # The target column in the training data

# Fit the model on the training data (train_data must include the 'price' column)
predictor = TabularPredictor(label=label_column).fit(
    train_data,
    presets='best_quality',
    hyperparameters={
        'XGB': {},  # XGBoost
        'GBM': {},  # LightGBM
 
    },
    time_limit=3600  # Optional time limit (in seconds)
)

# Make predictions on the test data (test_data should not have 'price')
predictions = predictor.predict(test_data)

In [ ]:
# Load the sample submission file
submission =pd.read_csv('/kaggle/input/playground-series-s4e9/sample_submission.csv')

# Replace the 'price' column in the submission file with the predictions
submission["price"] = predictions

# Save the submission file
submission.to_csv("submission.csv", index=False)

In [ ]:
submission.head()